# 🏗️ 기본 세팅

## MongoDB 서버 및 Python 패키지 설치

In [ ]:
# # 1. MongoDB 공식 GPG 키 및 리포지토리 등록
# !curl -fsSL https://www.mongodb.org/static/pgp/server-7.0.asc | sudo gpg --dearmor -o /usr/share/keyrings/mongodb-server-7.0.gpg
# !echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-7.0.gpg ] https://repo.mongodb.org/apt/ubuntu jammy/mongodb-org/7.0 multiverse" | sudo tee /etc/apt/sources.list.d/mongodb-org-7.0.list
#
# # 2. 패키지 목록 업데이트 및 설치
# !apt-get update
# !apt-get install -y mongodb-org > /dev/null
#
# # 3. MongoDB 서비스 실행
# !mkdir -p /data/db
# !mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db
#
# # 4. 파이썬용 드라이버 설치
# !pip install pymongo

# ✅ 실습문제

## 사전 세팅

In [1]:
from pymongo import MongoClient

# MongoDB Atlas 연결
client = MongoClient('mongodb://localhost:27017/')

db = client.health_db        # 원하는 DB 이름
col = db.health_records      # 원하는 컬렉션 이름

# 데이터 구성
data = [
    {"name": "David", "age": 45, "sex": "M", "systolic": 135, "diastolic": 88, "glucose": 110, "hdl": 40, "ldl": 150},
    {"name": "Alice", "age": 33, "sex": "F", "systolic": 118, "diastolic": 76, "glucose": 92, "hdl": 55, "ldl": 110},
    {"name": "Bob", "age": 52, "sex": "M", "systolic": 145, "diastolic": 95, "glucose": 160, "hdl": 38, "ldl": 170},
    {"name": "Emma", "age": 29, "sex": "F", "systolic": 109, "diastolic": 70, "glucose": 85, "hdl": 60, "ldl": 95},
    {"name": "Chris", "age": 60, "sex": "M", "systolic": 150, "diastolic": 92, "glucose": 170, "hdl": 35, "ldl": 190},
    {"name": "Sara", "age": 48, "sex": "F", "systolic": 132, "diastolic": 85, "glucose": 110, "hdl": 50, "ldl": 140}
]

# 데이터 삽입
col.insert_many(data)

print("데이터 삽입 완료!")


데이터 삽입 완료!


## 1. 고혈압 환자 찾기
- 수축기 혈압(sys) 140 이상

In [3]:
# 여기에 코드를 입력하세요.
result = col.find(
    {"systolic":{"$gte":140}}
)
display(list(result))


[{'_id': ObjectId('69e9da8823e020e8c6f6d08b'),
  'name': 'Bob',
  'age': 52,
  'sex': 'M',
  'systolic': 145,
  'diastolic': 95,
  'glucose': 160,
  'hdl': 38,
  'ldl': 170},
 {'_id': ObjectId('69e9da8823e020e8c6f6d08d'),
  'name': 'Chris',
  'age': 60,
  'sex': 'M',
  'systolic': 150,
  'diastolic': 92,
  'glucose': 170,
  'hdl': 35,
  'ldl': 190}]

## 2. 대사 질환 위험군 찾기
- 공복혈당 (glucose) 125 이상
- HDL 40 미만


In [9]:
# 여기에 코드를 입력하세요.
result_2 = col.find({
    "$and":[
        {"glucose":{"$gte":125}},
        {"hdl":{"$lt":40}}
    ]
})
display(list(result_2))

[{'_id': ObjectId('69e9da8823e020e8c6f6d08b'),
  'name': 'Bob',
  'age': 52,
  'sex': 'M',
  'systolic': 145,
  'diastolic': 95,
  'glucose': 160,
  'hdl': 38,
  'ldl': 170},
 {'_id': ObjectId('69e9da8823e020e8c6f6d08d'),
  'name': 'Chris',
  'age': 60,
  'sex': 'M',
  'systolic': 150,
  'diastolic': 92,
  'glucose': 170,
  'hdl': 35,
  'ldl': 190}]

## 3. 심혈관 위험도가 높은 사람 3명 찾기
- LDL 점수 = ldl 값 그대로
- 혈압 점수 = 수축기 혈압 sys
- 혈당 점수 = glucose

각 환자의 위험도(risk)는 다음의 합으로 계산합니다:
`risk = ldl + systolic + glucose`

Aggregation을 사용하여
1) 필요한 필드만 선택하고  
2) 새로운 필드(risk)를 생성하여 계산한 뒤  
3) risk 값을 기준으로 내림차순 정렬하여  

가장 위험도가 높은 3명을 출력하세요.


In [15]:
# 여기에 코드를 입력하세요.
import pandas as pd
result_3 = col.aggregate([
    {
        "$project":{
            "_id":0,
            "name":1,
            "age":1,
            "risk":{
                "$add":["$ldl","$systolic","$glucose"]
            }
        }
    },
    {"$sort":{"risk":-1}},
    {"$limit":3}
])

display(pd.DataFrame(list(result_3)))

,name,age,risk
0,Chris,60,510
1,Bob,52,475
2,David,45,395
